# Entrenar en la nube (Google Colab) sin descargar imágenes

Este notebook entrena el clasificador clásico usando la **caché de características**
compartida en Hugging Face Hub (~530 MB) en lugar de los ~10 GB de imágenes.

Requisitos:
1. Pertenecer a la organización / repos de Hugging Face del equipo (`configs/hub.yaml`).
2. En Colab: **🔑 Secrets → Add new secret** con nombre `HF_TOKEN` (token con permiso
   *write* si vas a publicar el modelo) y activar *Notebook access*.

Nunca pegues el token en una celda.

In [ ]:
!git clone --depth 1 https://github.com/juliovalenciag/clasifcador-fruta-o-verdura.git proyecto
%cd proyecto
!pip install -q -e .

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

## 1. Descargar la caché de características (no las imágenes)

In [ ]:
!frutas-data hub list-features
!frutas-data hub pull-features

## 2. Entrenar

Sin `data/processed/dataset.csv`, `--features` usa el manifiesto incluido en la caché.
La selección se hace en validation y test se evalúa una sola vez.

In [ ]:
!frutas-data train-classic --features

In [ ]:
import json, pandas as pd

metrics = json.load(open("artifacts/classical_metrics.json"))
print(metrics["selected_model"], json.dumps({k: metrics["test"][k] for k in ("accuracy", "balanced_accuracy", "f1_macro")}, indent=2))
pd.read_csv("artifacts/model_selection.csv")

## 3. Publicar el modelo para el equipo (opcional)

Solo si mejora al modelo vigente en **validation** (no elijas mirando test).
Usa una etiqueta de versión para que todos puedan fijarla en `configs/hub.yaml`.

In [ ]:
# !frutas-data hub push-model --tag v2.2.0